# Data Preprocessing Workflow

Preprocess `dataset/data.csv` including: handling missing values, encoding non-numeric data, preliminary statistical analysis, and visualization.

Ablation outputs:
- Original preprocessing workflow with the Section 4 encoding method: `data/processed_org/{train,val,test}.csv`
- Split-first preprocessing workflow with train-only imputation statistics and `pd.get_dummies`: `data/processed/{train,val,test}.csv`

## 1. Import packages

In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Load CSV raw data


Due to the large size, use `nrows`, read 5 lines to preview the structure, and then load the entire dataset to preprocess. 

In [13]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "preprocess" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "dataset" / "data.csv"
OUTPUT_PATH = PROJECT_ROOT / "dataset" / "processed_data.csv"
ORG_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed_org"
ALT_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

# Use nrows for preview
preview = pd.read_csv(DATA_PATH, sep=';', nrows=5)
print(f"Raw data path: {DATA_PATH}")
print(f"Rows: {preview.shape[0]},  Columns: {preview.shape[1]}")
print("\nColumn names:")
print(list(preview.columns))
preview

Raw data path: f:\2026_spring_project\2011_project\dataset\data.csv
Rows: 5,  Columns: 37

Column names:
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance\t', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular 

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [14]:
# Load the full dataset
df = pd.read_csv(DATA_PATH, sep=';')
print(f"Dataset shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

Dataset shape: (4424, 37)

Data types:
Marital status                                      int64
Application mode                                    int64
Application order                                   int64
Course                                              int64
Daytime/evening attendance\t                        int64
Previous qualification                              int64
Previous qualification (grade)                    float64
Nacionality                                         int64
Mother's qualification                              int64
Father's qualification                              int64
Mother's occupation                                 int64
Father's occupation                                 int64
Admission grade                                   float64
Displaced                                           int64
Educational special needs                           int64
Debtor                                              int64
Tuition fees up to date          

## 3. Handling Missing Values

In [15]:
# Count missing values and their proportions for each column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing Percentage (%)": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing Percentage (%)", ascending=False)

if missing_df.empty:
    print("No missing values in the dataset.")
else:
    print(f"Columns with missing values (total {len(missing_df)} columns):")
    print(missing_df)

No missing values in the dataset.


In [16]:
DROP_THRESHOLD = 50.0  # Drop columns if missing rate exceeds this threshold

# Remove columns with high missing rates
cols_to_drop = missing_df[missing_df["Missing Percentage (%)"] > DROP_THRESHOLD].index.tolist()
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped columns with missing rate > {DROP_THRESHOLD}%: {cols_to_drop}")

# Fill numeric columns with median, categorical columns with mode
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

for col in num_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Numeric column '{col}': filled missing values with median {median_val:.4f}")

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Categorical column '{col}': filled missing values with mode '{mode_val}'")

print(f"\nTotal missing values after processing: {df.isnull().sum().sum()}")


Total missing values after processing: 0


## 4. Handling Non-Numeric Data

Identify all string/categorical columns, apply `LabelEncoder` for encoding, and record the mappings.

In [17]:
# Re-detect non-numeric columns (may change after handling missing values)
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns: {cat_cols}")

label_encoders = {}
label_mappings = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"\nColumn '{col}' encoding mapping: {label_mappings[col]}")

print("\nEncoding completed. All columns are now numeric.")
print(f"Dataset shape: {df.shape}")

Non-numeric columns: ['Target']

Column 'Target' encoding mapping: {'Dropout': 0, 'Enrolled': 1, 'Graduate': 2}

Encoding completed. All columns are now numeric.
Dataset shape: (4424, 37)


## 5. Preliminary Statistical Analysis

In [ ]:
# Descriptive statistics (mean, variance, std, quantiles, etc.)
stats = df.describe().T
stats["variance"] = df.var()
stats["skewness"] = df.skew()
stats["kurtosis"] = df.kurtosis()
stats = stats[["count", "mean", "variance", "std", "min", "25%", "50%", "75%", "max", "skewness", "kurtosis"]]
stats.columns = ["Count", "Mean", "Variance", "Std Dev", "Min", "Q1", "Median", "Q3", "Max", "Skewness", "Kurtosis"]
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
stats

In [ ]:
# Correlation matrix (sorted by absolute value, display top 10 features with highest correlation to Target)
corr_matrix = df.corr()
if "Target" in corr_matrix.columns:
    target_corr = corr_matrix["Target"].drop("Target").abs().sort_values(ascending=False)
    print("Top 10 features with highest correlation to Target:")
    print(target_corr.head(10))

## 6. Data Distribution Visualization

In [ ]:
# Select continuous numeric features (treat low-cardinality columns as categorical, skip them)
CARDINALITY_THRESHOLD = 15
continuous_cols = [c for c in df.columns if df[c].nunique() > CARDINALITY_THRESHOLD]
print(f"Continuous features for plotting ({len(continuous_cols)} columns): {continuous_cols}")

# Histogram
n = len(continuous_cols)
if n > 0:
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3))
    axes = axes.flatten()
    for i, col in enumerate(continuous_cols):
        axes[i].hist(df[col].dropna(), bins=30, edgecolor='black', color='steelblue', alpha=0.7)
        axes[i].set_title(f"{col}", fontsize=9)
        axes[i].set_xlabel("Value")
        axes[i].set_ylabel("Frequency")
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Feature Distributions (Histogram)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plot (to visualize outliers)
if n > 0:
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3))
    axes = axes.flatten()
    for i, col in enumerate(continuous_cols):
        axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                        boxprops=dict(facecolor='lightblue', color='navy'),
                        medianprops=dict(color='red', linewidth=2))
        axes[i].set_title(f"{col}", fontsize=9)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Feature Distributions (Box Plot)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap (display top 15 features with highest correlation to Target)
top_features = corr_matrix["Target"].abs().sort_values(ascending=False).head(15).index.tolist() \
    if "Target" in corr_matrix.columns else list(df.columns[:15])

corr_sub = df[top_features].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_sub, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Correlation Heatmap (Top 15 Features)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Export Processed Data

In [18]:
from sklearn.model_selection import train_test_split

ORIGINAL_VAL_SIZE = 0.15
ORIGINAL_TEST_SIZE = 0.15
ORIGINAL_RANDOM_STATE = 42

original_stratify = df["Target"] if "Target" in df.columns else None
org_train_df, org_holdout_df = train_test_split(
    df,
    test_size=ORIGINAL_VAL_SIZE + ORIGINAL_TEST_SIZE,
    random_state=ORIGINAL_RANDOM_STATE,
    stratify=original_stratify,
    shuffle=True,
    )

org_holdout_test_ratio = ORIGINAL_TEST_SIZE / (ORIGINAL_VAL_SIZE + ORIGINAL_TEST_SIZE)
org_holdout_stratify = org_holdout_df["Target"] if "Target" in org_holdout_df.columns else None
org_val_df, org_test_df = train_test_split(
    org_holdout_df,
    test_size=org_holdout_test_ratio,
    random_state=ORIGINAL_RANDOM_STATE,
    stratify=org_holdout_stratify,
    shuffle=True,
    )

os.makedirs(ORG_OUTPUT_DIR, exist_ok=True)
org_train_df.to_csv(ORG_OUTPUT_DIR / "train.csv", index=False)
org_val_df.to_csv(ORG_OUTPUT_DIR / "val.csv", index=False)
org_test_df.to_csv(ORG_OUTPUT_DIR / "test.csv", index=False)

print("Original-method split sizes:")
print(f"  Train: {org_train_df.shape}")
print(f"  Val:   {org_val_df.shape}")
print(f"  Test:  {org_test_df.shape}")
print(f"Saved original-method splits under: {ORG_OUTPUT_DIR}")

org_train_df.head()

Original-method split sizes:
  Train: (3096, 37)
  Val:   (664, 37)
  Test:  (664, 37)
Saved original-method splits under: f:\2026_spring_project\2011_project\data\processed_org


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
3742,1,17,1,9670,1,1,118.0,1,19,38,...,0,6,6,6,14.000000,0,12.4,0.5,1.79,2
3539,1,1,1,9070,1,1,139.0,1,1,19,...,0,6,7,6,12.166667,1,16.2,0.3,-0.92,2
1117,1,1,1,9500,1,1,138.0,1,38,19,...,0,8,8,7,13.571429,0,13.9,-0.3,0.79,2
790,1,17,1,9773,1,1,138.0,1,1,19,...,0,6,6,6,13.833333,0,11.1,0.6,2.02,2
4380,1,7,1,9500,1,2,140.0,1,38,37,...,0,8,14,4,11.325000,1,12.7,3.7,-1.70,0


## 8. Split-First Preprocessing With Train Statistics

Alternative preprocessing pipeline:
- Split the raw dataset into train/validation/test first.
- Compute missing-value fill parameters from the training split only.
- Apply the training fill parameters to validation and test.
- Use `pd.get_dummies` for one-hot encoding and align validation/test columns to the training feature space.

In [19]:
import os

import pandas as pd
from sklearn.model_selection import train_test_split

ALT_DATA_PATH = PROJECT_ROOT / "dataset" / "data.csv"
TARGET_COL = "Target"
VAL_SIZE = 0.15
TEST_SIZE = 0.15
RANDOM_STATE = 42
CATEGORICAL_COLUMNS = {
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
    }

raw_df = pd.read_csv(ALT_DATA_PATH, sep=';')
raw_df.columns = raw_df.columns.str.strip()
print(f"Raw dataset path: {ALT_DATA_PATH}")
print(f"Raw dataset shape: {raw_df.shape}")

stratify_labels = raw_df[TARGET_COL] if TARGET_COL in raw_df.columns else None
train_df, holdout_df = train_test_split(
    raw_df,
    test_size=VAL_SIZE + TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_labels,
    shuffle=True,
    )

holdout_test_ratio = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
holdout_stratify = holdout_df[TARGET_COL] if TARGET_COL in holdout_df.columns else None
val_df, test_df = train_test_split(
    holdout_df,
    test_size=holdout_test_ratio,
    random_state=RANDOM_STATE,
    stratify=holdout_stratify,
    shuffle=True,
    )

feature_cols = [col for col in raw_df.columns if col != TARGET_COL]
categorical_cols = [col for col in feature_cols if col in CATEGORICAL_COLUMNS]
numeric_cols = [col for col in feature_cols if col not in categorical_cols]

numeric_fill_values = train_df[numeric_cols].median()
categorical_fill_values = {}
for col in categorical_cols:
    mode_values = train_df[col].mode(dropna=True)
    categorical_fill_values[col] = mode_values.iloc[0] if not mode_values.empty else "Missing"

def preprocess_split(split_df, train_feature_columns=None):
    features = split_df[feature_cols].copy()

    if numeric_cols:
        features.loc[:, numeric_cols] = features[numeric_cols].fillna(numeric_fill_values)

    for col, fill_value in categorical_fill_values.items():
        features[col] = features[col].fillna(fill_value)

    encoded_features = pd.get_dummies(features, columns=categorical_cols, dtype=int)

    if train_feature_columns is not None:
        encoded_features = encoded_features.reindex(columns=train_feature_columns, fill_value=0)

    return encoded_features

X_train = preprocess_split(train_df)
train_feature_columns = X_train.columns.tolist()
X_val = preprocess_split(val_df, train_feature_columns=train_feature_columns)
X_test = preprocess_split(test_df, train_feature_columns=train_feature_columns)

y_train = train_df[TARGET_COL].reset_index(drop=True) if TARGET_COL in train_df.columns else None
y_val = val_df[TARGET_COL].reset_index(drop=True) if TARGET_COL in val_df.columns else None
y_test = test_df[TARGET_COL].reset_index(drop=True) if TARGET_COL in test_df.columns else None

train_processed = X_train.reset_index(drop=True)
val_processed = X_val.reset_index(drop=True)
test_processed = X_test.reset_index(drop=True)

if y_train is not None:
    train_processed[TARGET_COL] = y_train
    val_processed[TARGET_COL] = y_val
    test_processed[TARGET_COL] = y_test

os.makedirs(ALT_OUTPUT_DIR, exist_ok=True)
train_processed.to_csv(ALT_OUTPUT_DIR / "train.csv", index=False)
val_processed.to_csv(ALT_OUTPUT_DIR / "val.csv", index=False)
test_processed.to_csv(ALT_OUTPUT_DIR / "test.csv", index=False)

print("Split sizes:")
print(f"  Train: {train_processed.shape}")
print(f"  Val:   {val_processed.shape}")
print(f"  Test:  {test_processed.shape}")
print(f"Categorical features one-hot encoded: {len(categorical_cols)}")
print(f"Encoded feature count: {len(train_feature_columns)}")
print(f"Missing values after preprocessing -> train: {train_processed.isna().sum().sum()}, val: {val_processed.isna().sum().sum()}, test: {test_processed.isna().sum().sum()}")
print(f"Saved files under: {ALT_OUTPUT_DIR}")

train_processed.head()

Raw dataset path: f:\2026_spring_project\2011_project\dataset\data.csv
Raw dataset shape: (4424, 37)
Split sizes:
  Train: (3096, 249)
  Val:   (664, 249)
  Test:  (664, 249)
Categorical features one-hot encoded: 18
Encoded feature count: 248
Missing values after preprocessing -> train: 0, val: 0, test: 0
Saved files under: f:\2026_spring_project\2011_project\data\processed


,Previous qualification (grade),Admission grade,Age at enrollment,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),...,Debtor_1,Tuition fees up to date_0,Tuition fees up to date_1,Gender_0,Gender_1,Scholarship holder_0,Scholarship holder_1,International_0,International_1,Target
0,118.0,113.5,19,0,6,9,5,13.500000,0,0,...,0,0,1,1,0,0,1,1,0,Graduate
1,139.0,134.9,20,0,6,7,6,13.666667,0,0,...,0,0,1,0,1,1,0,1,0,Graduate
2,138.0,144.3,20,0,7,9,6,12.700000,0,0,...,0,0,1,1,0,1,0,1,0,Graduate
3,138.0,124.0,19,0,6,6,6,13.166667,0,0,...,0,0,1,1,0,0,1,1,0,Graduate
4,140.0,140.0,29,0,8,14,4,11.325000,1,0,...,0,0,1,1,0,1,0,1,0,Dropout
